In [5]:
pip install Rouge

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install -U minsearch qdrant_client

  Attempting uninstall: qdrant_client
    Found existing installation: qdrant-client 1.14.3
    Uninstalling qdrant-client-1.14.3:
      Successfully uninstalled qdrant-client-1.14.3

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip show minsearch

Name: minsearch
Version: 0.0.4
Summary: Minimalistic text search engine that uses sklearn and pandas
Home-page: https://github.com/alexeygrigorev/minsearch
Author: 
Author-email: Alexey Grigorev <alexey@datatalks.club>
License: WTFPL
Location: /usr/local/python/3.12.1/lib/python3.12/site-packages
Requires: numpy, pandas, scikit-learn
Required-by: 
Note: you may need to restart the kernel to use updated packages.


# QUESTION 1

In [6]:
import requests
import pandas as pd
from tqdm.auto import tqdm
import minsearch

# 🔹 Baixar os dados
url_prefix = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/'
docs_url = url_prefix + 'search_evaluation/documents-with-ids.json'
documents = requests.get(docs_url).json()

ground_truth_url = url_prefix + 'search_evaluation/ground-truth-data.csv'
df_ground_truth = pd.read_csv(ground_truth_url)
ground_truth = df_ground_truth.to_dict(orient='records')

# 🔹 Métricas de avaliação
def hit_rate(relevance_total):
    return sum(True in r for r in relevance_total) / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0
    for r in relevance_total:
        for i, rel in enumerate(r):
            if rel:
                total_score += 1 / (i + 1)
                break
    return total_score / len(relevance_total)

def evaluate(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        doc_id = q['document']
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)
    return {'hit_rate': hit_rate(relevance_total), 'mrr': mrr(relevance_total)}

# 🔹 Criação do índice sem boost (será aplicado na busca)
index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)
index.fit(documents)

# 🔹 Parâmetros de boost aplicados diretamente na busca
boost = {"question": 1.5, "section": 0.1}

def search(q):
    return index.search(
        query=q["question"],
        filter_dict={"course": q["course"]},
        boost_dict=boost,
        num_results=5
    )

# 🔹 Executar avaliação
results = evaluate(ground_truth, search)
print("Resultado Questão 1:")
print(results)


  0%|          | 0/4627 [00:00<?, ?it/s]

Resultado Questão 1:
{'hit_rate': 0.848714069591528, 'mrr': 0.7283553058137033}


# QUESTION 2

In [7]:
# Questão 2: Busca vetorial com minsearch usando apenas o campo 'question'

from minsearch import VectorSearch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline

# 🔹 Coletar os textos das perguntas
texts = [doc['question'] for doc in documents]

# 🔹 Criar pipeline de embeddings (TF-IDF + SVD)
pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)

# 🔹 Gerar embeddings
X = pipeline.fit_transform(texts)

# 🔹 Criar índice vetorial
vindex = VectorSearch(keyword_fields={"course"})
vindex.fit(X, documents)

# 🔹 Função de busca
def search_vector(q):
    vec = pipeline.transform([q["question"]])
    return vindex.search(
        query_vector=vec[0],
        filter_dict={"course": q["course"]},
        num_results=5
    )

# 🔹 Avaliar
results_vector_q = evaluate(ground_truth, search_vector)

print("Resultado Questão 2:")
print(results_vector_q)

  0%|          | 0/4627 [00:00<?, ?it/s]

Resultado Questão 2:
{'hit_rate': 0.48173762697212014, 'mrr': 0.3568510914199265}


# QUESTION 3

In [10]:
import requests
import pandas as pd
from tqdm.auto import tqdm
import numpy as np

from minsearch import VectorSearch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline

# 🔹 Baixar os dados
url_prefix = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/'
docs_url = url_prefix + 'search_evaluation/documents-with-ids.json'
documents = requests.get(docs_url).json()

ground_truth_url = url_prefix + 'search_evaluation/ground-truth-data.csv'
df_ground_truth = pd.read_csv(ground_truth_url)
ground_truth = df_ground_truth.to_dict(orient='records')

# 🔹 Métricas
def hit_rate(relevance_total):
    return sum(True in r for r in relevance_total) / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0
    for r in relevance_total:
        for i, rel in enumerate(r):
            if rel:
                total_score += 1 / (i + 1)
                break
    return total_score / len(relevance_total)

def evaluate(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        doc_id = q['document']
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)
    return {'hit_rate': hit_rate(relevance_total), 'mrr': mrr(relevance_total)}

# 🔹 Vetor com question + text
texts_q3 = [doc["question"] + " " + doc["text"] for doc in documents]

pipeline_q3 = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)

X_q3 = pipeline_q3.fit_transform(texts_q3)

# 🔹 Corrigido: keyword_fields exigido pelo VectorSearch
vindex_q3 = VectorSearch(keyword_fields=["course"])
vindex_q3.fit(X_q3, documents)

def search_q3(q):
    X_query = pipeline_q3.transform([q["question"]])
    return vindex_q3.search(X_query[0], num_results=5)

# 🔹 Avaliação
results_q3 = evaluate(ground_truth, search_q3)
print("Resultado Questão 3:")
print(results_q3)


  0%|          | 0/4627 [00:00<?, ?it/s]

Resultado Questão 3:
{'hit_rate': 0.7704776312945754, 'mrr': 0.6150097255240982}


# QUESTION 4

In [11]:
pip install -q qdrant-client[fastembed]


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
pip install -U fastembed


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
pip install -q 'fastembed[jina]'


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
import uuid
import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

from fastembed import TextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

# 🔹 Ajustes para evitar crashes
os.environ["ORT_DISABLE_MEMORY_ARENA"] = "1"
os.environ["OMP_NUM_THREADS"] = "2"

# 🔹 Baixar dados
url_prefix = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/"
docs_url = url_prefix + "search_evaluation/documents-with-ids.json"
documents = requests.get(docs_url).json()

ground_truth_url = url_prefix + "search_evaluation/ground-truth-data.csv"
df_ground_truth = pd.read_csv(ground_truth_url)
ground_truth = df_ground_truth.to_dict(orient="records")

# 🔹 Preparar textos para embedding
for doc in documents:
    doc["text_combined"] = doc["question"] + " " + doc["text"]

texts = [doc["text_combined"] for doc in documents]

# 🔹 Criar embeddings com batching
embedding_model = TextEmbedding(model_name="jinaai/jina-embeddings-v2-small-en")

def embed_in_batches(texts, batch_size=8):
    vectors = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding batches"):
        batch = texts[i:i + batch_size]
        vectors.extend(embedding_model.embed(batch, batch_size=batch_size))
    return np.array(vectors)

vectors = embed_in_batches(texts)

# 🔹 Inicializar Qdrant local (em memória)
client = QdrantClient(":memory:")

# 🔹 Criar coleção (com check se já existe)
if client.collection_exists("faq"):
    client.delete_collection("faq")

client.create_collection(
    collection_name="faq",
    vectors_config=VectorParams(size=vectors.shape[1], distance=Distance.COSINE)
)

# ✅ Usar UUIDs válidos e manter o campo "id" original no payload
points = []
for vec, doc in zip(vectors, documents):
    doc["original_id"] = doc["id"]
    point_id = str(uuid.uuid4())
    points.append(PointStruct(id=point_id, vector=vec.tolist(), payload=doc))

client.upsert(collection_name="faq", points=points)

# 🔹 Funções de avaliação
def hit_rate(relevance_total):
    return sum(True in r for r in relevance_total) / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0
    for r in relevance_total:
        for i, rel in enumerate(r):
            if rel:
                total_score += 1 / (i + 1)
                break
    return total_score / len(relevance_total)

def evaluate(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        doc_id = q["document"]
        results = search_function(q)
        relevance = [str(hit.payload.get("original_id")) == str(doc_id) for hit in results]
        relevance_total.append(relevance)
    return {'hit_rate': hit_rate(relevance_total), 'mrr': mrr(relevance_total)}

# 🔹 Função de busca usando `.search()` com vetor diretamente
def search_qdrant(q):
    query = q["question"] + " " + q.get("answer", "")
    query_vector = list(embedding_model.embed([query]))[0]
    return client.search(
        collection_name="faq",
        query_vector=query_vector.tolist(),
        limit=5,
        with_payload=True
    )

# 🔹 Avaliação
results_q4 = evaluate(ground_truth, search_qdrant)
print("Resultado Questão 4:")
print(results_q4)


Embedding batches:   0%|          | 0/119 [00:00<?, ?it/s]

  0%|          | 0/4627 [00:00<?, ?it/s]

/tmp/ipykernel_32272/2669450199.py:90: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  return client.search(


Resultado Questão 4:
{'hit_rate': 0.9120380376053598, 'mrr': 0.8240040342914787}


# QUESTION 5

In [3]:
import os
import uuid
import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

from fastembed import TextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

# Questão 5 - Cosine Similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline

# 🔹 Ajustes para evitar crashes
os.environ["ORT_DISABLE_MEMORY_ARENA"] = "1"
os.environ["OMP_NUM_THREADS"] = "2"

# 🔹 Baixar dados
url_prefix = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/"
docs_url = url_prefix + "search_evaluation/documents-with-ids.json"
documents = requests.get(docs_url).json()

ground_truth_url = url_prefix + "search_evaluation/ground-truth-data.csv"
df_ground_truth = pd.read_csv(ground_truth_url)
ground_truth = df_ground_truth.to_dict(orient="records")

# 🔹 Preparar textos para embedding
for doc in documents:
    doc["text_combined"] = doc["question"] + " " + doc["text"]

texts = [doc["text_combined"] for doc in documents]

# 🔹 Criar embeddings com batching
embedding_model = TextEmbedding(model_name="jinaai/jina-embeddings-v2-small-en")

def embed_in_batches(texts, batch_size=8):
    vectors = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding batches"):
        batch = texts[i:i + batch_size]
        vectors.extend(embedding_model.embed(batch, batch_size=batch_size))
    return np.array(vectors)

vectors = embed_in_batches(texts)

# 🔹 Inicializar Qdrant local (em memória)
client = QdrantClient(":memory:")

# 🔹 Criar coleção (com check se já existe)
if client.collection_exists("faq"):
    client.delete_collection("faq")

client.create_collection(
    collection_name="faq",
    vectors_config=VectorParams(size=vectors.shape[1], distance=Distance.COSINE)
)

# ✅ Usar UUIDs válidos e manter o campo "id" original no payload
points = []
for vec, doc in zip(vectors, documents):
    doc["original_id"] = doc["id"]
    point_id = str(uuid.uuid4())
    points.append(PointStruct(id=point_id, vector=vec.tolist(), payload=doc))

client.upsert(collection_name="faq", points=points)

# 🔹 Função de busca (usando 'search' para compatibilidade com versão local)
def search_qdrant(q):
    query = q["question"] + " " + q.get("answer", "")
    query_vector = list(embedding_model.embed([query]))[0]
    return client.search(
        collection_name="faq",
        query_vector=query_vector.tolist(),
        limit=5,
        with_payload=True
    )

# 🔹 Questão 5: Cosine Similarity entre resposta LLM e resposta original
results_url = url_prefix + 'rag_evaluation/data/results-gpt4o-mini.csv'
df_results = pd.read_csv(results_url)

# Criar pipeline de vetorização
pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)

# Treinar o pipeline com todo o conteúdo textual
all_text = (
    df_results["answer_llm"].fillna('') + " " +
    df_results["answer_orig"].fillna('') + " " +
    df_results["question"].fillna('')
)
pipeline.fit(all_text)

# Criar os vetores para respostas
v_llm = pipeline.transform(df_results["answer_llm"].fillna(''))
v_orig = pipeline.transform(df_results["answer_orig"].fillna(''))

# Função de cosine similarity
def cosine(u, v):
    u_norm = np.sqrt((u * u).sum(axis=1))
    v_norm = np.sqrt((v * v).sum(axis=1))
    dot = (u * v).sum(axis=1)
    return dot / (u_norm * v_norm)

cosine_similarities = cosine(v_llm, v_orig)
average_cosine = cosine_similarities.mean()

print("Resultado Questão 5: Cosine similarity média entre respostas LLM e respostas originais")
print("Average cosine similarity:", round(average_cosine, 4))


Embedding batches:   0%|          | 0/119 [00:00<?, ?it/s]

Resultado Questão 5: Cosine similarity média entre respostas LLM e respostas originais
Average cosine similarity: 0.8416


# QUESTION 6

In [4]:
import os
import uuid
import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

from fastembed import TextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

# Questão 5 - Cosine Similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline

# Questão 6 - ROUGE
from rouge import Rouge

# 🔹 Ajustes para evitar crashes
os.environ["ORT_DISABLE_MEMORY_ARENA"] = "1"
os.environ["OMP_NUM_THREADS"] = "2"

# 🔹 Baixar dados
url_prefix = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/"
docs_url = url_prefix + "search_evaluation/documents-with-ids.json"
documents = requests.get(docs_url).json()

ground_truth_url = url_prefix + "search_evaluation/ground-truth-data.csv"
df_ground_truth = pd.read_csv(ground_truth_url)
ground_truth = df_ground_truth.to_dict(orient="records")

# 🔹 Preparar textos para embedding
for doc in documents:
    doc["text_combined"] = doc["question"] + " " + doc["text"]

texts = [doc["text_combined"] for doc in documents]

# 🔹 Criar embeddings com batching
embedding_model = TextEmbedding(model_name="jinaai/jina-embeddings-v2-small-en")

def embed_in_batches(texts, batch_size=8):
    vectors = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding batches"):
        batch = texts[i:i + batch_size]
        vectors.extend(embedding_model.embed(batch, batch_size=batch_size))
    return np.array(vectors)

vectors = embed_in_batches(texts)

# 🔹 Inicializar Qdrant local (em memória)
client = QdrantClient(":memory:")

# 🔹 Criar coleção (com check se já existe)
if client.collection_exists("faq"):
    client.delete_collection("faq")

client.create_collection(
    collection_name="faq",
    vectors_config=VectorParams(size=vectors.shape[1], distance=Distance.COSINE)
)

# ✅ Usar UUIDs válidos e manter o campo "id" original no payload
points = []
for vec, doc in zip(vectors, documents):
    doc["original_id"] = doc["id"]
    point_id = str(uuid.uuid4())
    points.append(PointStruct(id=point_id, vector=vec.tolist(), payload=doc))

client.upsert(collection_name="faq", points=points)

# 🔹 Função de busca (usando 'search' para compatibilidade com versão local)
def search_qdrant(q):
    query = q["question"] + " " + q.get("answer", "")
    query_vector = list(embedding_model.embed([query]))[0]
    return client.search(
        collection_name="faq",
        query_vector=query_vector.tolist(),
        limit=5,
        with_payload=True
    )

# 🔹 Questão 5: Cosine Similarity entre resposta LLM e resposta original
results_url = url_prefix + 'rag_evaluation/data/results-gpt4o-mini.csv'
df_results = pd.read_csv(results_url)

# Criar pipeline de vetorização
pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)

# Treinar o pipeline com todo o conteúdo textual
all_text = (
    df_results["answer_llm"].fillna('') + " " +
    df_results["answer_orig"].fillna('') + " " +
    df_results["question"].fillna('')
)
pipeline.fit(all_text)

# Criar os vetores para respostas
v_llm = pipeline.transform(df_results["answer_llm"].fillna(''))
v_orig = pipeline.transform(df_results["answer_orig"].fillna(''))

# Função de cosine similarity
def cosine(u, v):
    u_norm = np.sqrt((u * u).sum(axis=1))
    v_norm = np.sqrt((v * v).sum(axis=1))
    dot = (u * v).sum(axis=1)
    return dot / (u_norm * v_norm)

cosine_similarities = cosine(v_llm, v_orig)
average_cosine = cosine_similarities.mean()

print("Resultado Questão 5: Cosine similarity média entre respostas LLM e respostas originais")
print("Average cosine similarity:", round(average_cosine, 4))

# 🔹 Questão 6: ROUGE-1 F1 Score médio entre respostas LLM e originais
rouge = Rouge()
rouge_1_f1_scores = []

for _, row in tqdm(df_results.iterrows(), total=len(df_results), desc="Calculando ROUGE-1 F1"):
    try:
        scores = rouge.get_scores(row["answer_llm"], row["answer_orig"])[0]
        rouge_1_f1 = scores["rouge-1"]["f"]
    except ValueError:
        rouge_1_f1 = 0.0
    rouge_1_f1_scores.append(rouge_1_f1)

avg_rouge_1_f1 = np.mean(rouge_1_f1_scores)

print("Resultado Questão 6: ROUGE-1 F1 médio entre respostas LLM e respostas originais")
print("Average ROUGE-1 F1 Score:", round(avg_rouge_1_f1, 4))


Embedding batches:   0%|          | 0/119 [00:00<?, ?it/s]

Resultado Questão 5: Cosine similarity média entre respostas LLM e respostas originais
Average cosine similarity: 0.8416


Calculando ROUGE-1 F1:   0%|          | 0/1830 [00:00<?, ?it/s]

Resultado Questão 6: ROUGE-1 F1 médio entre respostas LLM e respostas originais
Average ROUGE-1 F1 Score: 0.3517
